## Feature Engineering

### By: Maricel Martinez

### Date: 2026-08-27

### Description:

Proceso de Feature Engineering (Issue 4) para limpiar, transformar y preparar los datos del proyecto `Precios-casas-Boston` para el entrenamiento de un modelo, usando transformadores y pipelines de scikit-learn.

### Alcance

Este notebook parte de los hallazgos de los análisis univariable (`03_EDA_univariable`), bivariable (`04_EDA_bivariable`) y multivariable (`05_EDA_multivariable`) ya realizados. Se desarrollan los cinco pasos de la Issue 4:

1. Limpieza de datos
2. Selección de atributos (Feature Selection)
3. Ingeniería de atributos (Feature Engineering)
4. Escalado de atributos (Feature Scaling)
5. Encoding

### Nota de flujo de trabajo

Este notebook debe desarrollarse en una rama nueva siguiendo Gitflow (por ejemplo, `feature/04-feature-engineering`), y su incorporación a `main` requiere Pull Request con al menos una revisión y el paso de los checks de CI/CD, según lo establece la Issue 4.

## 📚 Import libraries

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn as sk
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    KBinsDiscretizer,
    OneHotEncoder,
    StandardScaler,
)

# Configuración y versiones

In [2]:
pd.set_option("display.float_format", "{:.2f}".format)

print("Pandas version:", pd.__version__)
print("sklearn version:", sk.__version__)

Pandas version: 3.0.5
sklearn version: 1.9.0


## 💾 Load data

In [3]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

boston_df = pd.read_parquet(
    DATA_DIR / "02_intermediate/boston_type_fixed.parquet", engine="pyarrow"
)

# 👷 Preparación de datos

## Selección inicial de columnas

A diferencia del dataset de Titanic (que tenía una columna `name` de texto libre sin valor predictivo), todas las columnas de este dataset son numéricas y relevantes según el análisis exploratorio previo. Se mantienen todas para la etapa de limpieza; los descartes por relevancia se abordan en la sección de Feature Selection.

In [4]:
selected_features = [
    "crim", "zn", "indus", "chas", "nox", "rm", "age",
    "dis", "rad", "tax", "ptratio", "black", "lstat", "medv",
]

boston_features = boston_df[selected_features].copy()
boston_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 343 entries, 0 to 342
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     339 non-null    float64
 1   zn       343 non-null    float64
 2   indus    339 non-null    float64
 3   chas     343 non-null    float64
 4   nox      341 non-null    float64
 5   rm       340 non-null    float64
 6   age      342 non-null    float64
 7   dis      340 non-null    float64
 8   rad      342 non-null    float64
 9   tax      340 non-null    float64
 10  ptratio  342 non-null    float64
 11  black    341 non-null    float64
 12  lstat    343 non-null    float64
 13  medv     331 non-null    float64
dtypes: float64(14)
memory usage: 37.6 KB


## 🧹 Limpieza de datos

### Valores nulos

In [5]:
boston_features.isna().sum()

crim        4
zn          0
indus       4
chas        0
nox         2
rm          3
age         1
dis         3
rad         1
tax         3
ptratio     1
black       2
lstat       0
medv       12
dtype: int64

#### Interpretación de los valores nulos

Se confirman los porcentajes de nulos ya identificados en el análisis univariable: `medv` es la variable con más nulos (12 de 343, 3.50 %), seguida de `crim` e `indus` (4 cada una). Las variables predictoras se imputarán dentro de los pipelines (mediana para continuas, moda para categóricas). La variable objetivo `medv` no puede imputarse: sus filas con valor nulo se eliminan a continuación, ya que no pueden usarse para entrenar ni evaluar un modelo supervisado.

### Filas duplicadas

In [6]:
duplicate_rows = boston_features.duplicated().sum()
print("Número de filas duplicadas:", duplicate_rows)

boston_features = boston_features.drop_duplicates()
boston_features.info()

Número de filas duplicadas: 10
<class 'pandas.DataFrame'>
RangeIndex: 333 entries, 0 to 332
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     329 non-null    float64
 1   zn       333 non-null    float64
 2   indus    329 non-null    float64
 3   chas     333 non-null    float64
 4   nox      331 non-null    float64
 5   rm       330 non-null    float64
 6   age      332 non-null    float64
 7   dis      330 non-null    float64
 8   rad      332 non-null    float64
 9   tax      330 non-null    float64
 10  ptratio  332 non-null    float64
 11  black    331 non-null    float64
 12  lstat    333 non-null    float64
 13  medv     321 non-null    float64
dtypes: float64(14)
memory usage: 36.6 KB


#### Interpretación de duplicados

Se eliminaron 10 filas completamente duplicadas, consistente con lo encontrado en el análisis univariable. El dataset queda en 333 filas. Esto reduce el riesgo de que el modelo sobreajuste a observaciones repetidas.

### Valores faltantes en la variable objetivo (medv)

In [7]:
print("Filas con medv nulo:", boston_features["medv"].isna().sum())

boston_features = boston_features.dropna(subset=["medv"])
boston_features.info()

Filas con medv nulo: 12
<class 'pandas.DataFrame'>
Index: 321 entries, 0 to 332
Data columns (total 14 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     319 non-null    float64
 1   zn       321 non-null    float64
 2   indus    317 non-null    float64
 3   chas     321 non-null    float64
 4   nox      319 non-null    float64
 5   rm       319 non-null    float64
 6   age      320 non-null    float64
 7   dis      319 non-null    float64
 8   rad      320 non-null    float64
 9   tax      319 non-null    float64
 10  ptratio  320 non-null    float64
 11  black    319 non-null    float64
 12  lstat    321 non-null    float64
 13  medv     321 non-null    float64
dtypes: float64(14)
memory usage: 37.6 KB


#### Interpretación

Se eliminaron 12 filas sin valor de `medv`, ya que no aportan información para el entrenamiento supervisado y no es apropiado imputar la variable objetivo. El dataset queda en 321 filas, que serán la base para el resto del proceso de Feature Engineering.

### Revisión de valores atípicos: posible censura en medv

El análisis bivariable y multivariable detectó que `medv = 50.00` aparece de forma repetida y desconectada del comportamiento esperado según las variables predictoras (11 observaciones), lo que sugiere un valor censurado más que un precio real observado.

Siguiendo la Issue 4 ("los valores atípicos pueden separarse del dataset dependiendo del problema del proyecto"), se decide **no eliminar** estas filas por ahora — perderíamos información real de las viviendas más caras — sino **documentarlas** con una columna indicadora, para que puedan excluirse fácilmente durante el modelamiento si se decide tratarlas por separado.

In [8]:
boston_features["medv_censored"] = (boston_features["medv"] == 50.0).astype(int)

print("Observaciones marcadas como posiblemente censuradas:", boston_features["medv_censored"].sum())

Observaciones marcadas como posiblemente censuradas: 11


# 🎯 Selección de atributos (Feature Selection)

El análisis bivariable encontró una correlación de 0.91 entre `tax` y `rad` — la más alta del dataset entre dos predictoras — y el análisis multivariable confirmó que ambas están dominadas por el mismo grupo de observaciones (`rad = 24`, con `tax` prácticamente constante en 666). Mantener ambas variables introduce redundancia y riesgo de inestabilidad en modelos lineales.

**Decisión:** se descarta `tax` y se conserva `rad`, ya que `rad` es la variable con la que se construirá además un atributo derivado (`rad_group`) en la siguiente sección, aportando la misma información de forma más interpretable.

In [9]:
boston_features = boston_features.drop(columns=["tax"])

duplicate_after_selection = boston_features.duplicated().sum()
print("Filas duplicadas tras eliminar tax:", duplicate_after_selection)
boston_features.columns.tolist()

Filas duplicadas tras eliminar tax: 0


['crim',
 'zn',
 'indus',
 'chas',
 'nox',
 'rm',
 'age',
 'dis',
 'rad',
 'ptratio',
 'black',
 'lstat',
 'medv',
 'medv_censored']

#### Interpretación

Al eliminar `tax` no aparecieron nuevas filas duplicadas (0), por lo que no fue necesario un `drop_duplicates()` adicional. El dataset queda con 13 columnas (12 predictoras + `medv_censored` + `medv`).

# 🛠️ Ingeniería de atributos (Feature Engineering)

### Nuevo atributo: rad_group

El análisis bivariable y multivariable mostró que el grupo `rad = 24` (el más numeroso) se comporta de forma sistemáticamente distinta al resto: precios más bajos y `tax` casi constante. Se crea un atributo binario que captura esta distinción de forma explícita, complementando a `rad` en vez de reemplazarlo.

In [10]:
boston_features["rad_group"] = np.where(
    boston_features["rad"] == 24, "alto", "bajo"
)

boston_features["rad_group"].value_counts()

rad_group
bajo    234
alto     87
Name: count, dtype: int64

#### Interpretación de rad_group

La distribución confirma lo esperado: 87 observaciones quedan en el grupo `alto` (`rad = 24`) y 234 en `bajo` (el resto de valores de `rad`), sobre un total de 321 filas. La proporción (≈27 % vs ≈73 %) es lo suficientemente balanceada como para que ambos grupos estén bien representados tanto en entrenamiento como en prueba.

### Transformaciones y discretización dentro del pipeline

Según la Issue 4, se debe discretizar al menos una variable continua y evaluar transformaciones prometedoras (log, sqrt, etc.). Estas se implementan como transformadores de scikit-learn (no manualmente con pandas), para que formen parte reproducible del pipeline:

- **Discretización de `age`**: se agrega como atributo adicional `age` dividida en 4 categorías (`KBinsDiscretizer`), complementando a la versión continua escalada.
- **Transformación logarítmica de `crim`**: es la variable más asimétrica del dataset (skewness = 4.64, kurtosis = 31.47, según el análisis univariable). Se aplica `log1p` para reducir su asimetría antes de escalar.

# ⚖️ Escalado y 🔤 Encoding — definición de pipelines

Se agrupan las columnas por el tratamiento que necesitan y se define un pipeline de scikit-learn para cada grupo. El escalado (`StandardScaler`) se incluye explícitamente para todas las variables numéricas — este paso no estaba implementado en el ejemplo del profesor y es requisito de la Issue 4.

In [11]:
cols_log = ["crim"]

cols_numeric = ["zn", "indus", "nox", "rm", "age", "dis", "ptratio", "black", "lstat"]

cols_discretize = ["age"]

cols_binary = ["chas", "medv_censored"]

cols_nominal = ["rad", "rad_group"]

print("Log + escalado:", cols_log)
print("Numéricas continuas:", cols_numeric)
print("Discretizada (además de continua):", cols_discretize)
print("Binarias:", cols_binary)
print("Categóricas nominales:", cols_nominal)

Log + escalado: ['crim']
Numéricas continuas: ['zn', 'indus', 'nox', 'rm', 'age', 'dis', 'ptratio', 'black', 'lstat']
Discretizada (además de continua): ['age']
Binarias: ['chas', 'medv_censored']
Categóricas nominales: ['rad', 'rad_group']


In [12]:
# Pipeline: variable con transformación logarítmica antes de escalar
log_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler()),
    ]
)

# Pipeline: variables numéricas continuas
numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

# Pipeline: discretización de age en 4 categorías (nuevo atributo)
discretize_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("discretizer", KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile")),
    ]
)

# Pipeline: variables binarias (ya en 0/1, solo se imputan por seguridad)
binary_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ]
)

# Pipeline: variables categóricas nominales
# rad y rad_group se tratan como nominales (no ordinales): el análisis
# multivariable mostró que el efecto de rad sobre medv no es monótono,
# está dominado por el grupo rad = 24, así que imponer un orden numérico
# sería una suposición incorrecta.
nominal_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

In [13]:
preprocessor = ColumnTransformer(
    transformers=[
        ("log", log_pipe, cols_log),
        ("numeric", numeric_pipe, cols_numeric),
        ("discretize", discretize_pipe, cols_discretize),
        ("binary", binary_pipe, cols_binary),
        ("nominal", nominal_pipe, cols_nominal),
    ]
)
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log', ...), ('numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featu

#### Interpretación del diseño del preprocesador

`age` aparece en dos ramas del `ColumnTransformer` a propósito: una vez escalada como continua (rama `numeric`) y otra vez discretizada en 4 categorías (rama `discretize`), generando dos atributos distintos a partir de la misma columna original, tal como pide la Issue 4 ("agregar transformaciones prometedoras" sin necesariamente reemplazar el atributo original).

`crim` recibe un tratamiento especial (log + escalado) por ser, de lejos, la variable más asimétrica del dataset. El resto de variables continuas solo requiere imputación y escalado, ya que su asimetría es moderada o baja.

`chas` y `medv_censored` ya son binarias (0/1) y no requieren codificación adicional, solo imputación por seguridad. `rad` y `rad_group` se codifican como nominales con One-Hot Encoding.

# Train / Test split

In [14]:
X_features = boston_features.drop(columns=["medv"])
y_target = boston_features["medv"]

# 80% train, 20% test
# se estratifica por chas: es una variable binaria con un grupo muy
# pequeño (35 observaciones en todo el dataset, según el univariable),
# por lo que sin estratificar se corre el riesgo de que quede
# subrepresentado en alguno de los dos conjuntos
x_train, x_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.2, stratify=X_features["chas"], random_state=42
)

x_train.shape, y_train.shape

((256, 14), (256,))

In [15]:
x_test.shape, y_test.shape

((65, 14), (65,))

# Preprocessing pipeline

In [16]:
preprocessor.fit(x_train)
feature_names = preprocessor.get_feature_names_out()

x_train_transformed = preprocessor.transform(x_train)
x_train_transformed = pd.DataFrame(x_train_transformed, columns=feature_names)
x_train_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 256 entries, 0 to 255
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   log__crim                256 non-null    float64
 1   numeric__zn              256 non-null    float64
 2   numeric__indus           256 non-null    float64
 3   numeric__nox             256 non-null    float64
 4   numeric__rm              256 non-null    float64
 5   numeric__age             256 non-null    float64
 6   numeric__dis             256 non-null    float64
 7   numeric__ptratio         256 non-null    float64
 8   numeric__black           256 non-null    float64
 9   numeric__lstat           256 non-null    float64
 10  discretize__age          256 non-null    float64
 11  binary__chas             256 non-null    float64
 12  binary__medv_censored    256 non-null    float64
 13  nominal__rad_1.0         256 non-null    float64
 14  nominal__rad_2.0         256 non-null

In [17]:
x_train_transformed.head()

,log__crim,numeric__zn,numeric__indus,numeric__nox,numeric__rm,numeric__age,numeric__dis,numeric__ptratio,numeric__black,numeric__lstat,...,nominal__rad_2.0,nominal__rad_3.0,nominal__rad_4.0,nominal__rad_5.0,nominal__rad_6.0,nominal__rad_7.0,nominal__rad_8.0,nominal__rad_24.0,nominal__rad_group_alto,nominal__rad_group_bajo
0,-0.74,2.22,-1.33,-1.31,-0.53,-1.78,3.56,-0.09,0.37,-0.66,...,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00
1,-0.68,-0.46,-0.99,-0.40,-0.96,0.74,-0.57,-0.90,0.43,0.30,...,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00
2,-0.74,-0.46,-0.99,-0.40,-0.35,-0.75,-0.08,-0.90,0.38,-0.34,...,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00
3,-0.06,-0.46,-0.40,-0.14,-0.77,0.94,0.27,1.20,-0.00,1.39,...,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00
4,-0.65,-0.46,-0.71,-0.50,-0.42,-1.36,0.07,0.34,0.39,-0.33,...,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,0.00,1.00


### Interpretación del resultado del pipeline

El pipeline transformó correctamente los datos: `x_train_transformed` tiene 256 filas (80 % de las 321 originales, tras el `train_test_split` estratificado por `chas`) y **24 columnas**, sin ningún valor nulo. El conteo de columnas es consistente con lo diseñado:

- 1 columna `log__crim` (log + escalado)
- 9 columnas `numeric__*` (continuas escaladas, incluye `age` escalada)
- 1 columna `discretize__age` (la misma `age`, discretizada en 4 categorías ordinales — el segundo atributo derivado de esta variable)
- 2 columnas `binary__*` (`chas`, `medv_censored`)
- 9 columnas `nominal__rad_*` (One-Hot de las 9 categorías originales de `rad`, incluyendo `rad_24.0`)
- 2 columnas `nominal__rad_group_*` (One-Hot de `alto`/`bajo`)

Las variables escaladas (`log__crim`, `numeric__*`) muestran valores centrados alrededor de 0 con magnitudes típicas entre -2 y 2, consistente con un `StandardScaler` aplicado correctamente. El conjunto de prueba (`x_test`, 65 filas) sigue pendiente de transformar con el mismo `preprocessor` ya ajustado (`preprocessor.transform(x_test)`), usando `.transform()` y no `.fit_transform()`, para evitar fuga de información del conjunto de entrenamiento.

## 📊 Análisis de resultados y conclusiones

El proceso de Feature Engineering partió de 343 filas y 14 columnas, y terminó en un conjunto de entrenamiento (`x_train_transformed`) de 256 filas y 24 columnas, completamente numérico y sin valores nulos, listo para modelar.

**Limpieza:** se eliminaron 10 filas duplicadas y 12 filas sin valor de `medv` (variable objetivo), quedando el dataset base en 321 filas. Las 11 observaciones con `medv = 50.00` (posible censura, identificada desde el análisis univariable) no se eliminaron, sino que se documentaron con la columna `medv_censored`, preservando la información pero dejándola disponible para excluirse en el modelamiento si se decide tratarla aparte.

**Feature Selection:** se eliminó `tax` por su alta colinealidad con `rad` (0.91, detectada en el bivariable), sin generar filas duplicadas adicionales.

**Feature Engineering:** se creó `rad_group` (87 observaciones en `alto` / 234 en `bajo`), capturando de forma simplificada el mismo patrón que ya se había detectado en el análisis bivariable y multivariable para el grupo `rad = 24`. Además, `age` se transformó de dos formas distintas (escalada como continua y discretizada en 4 categorías), y `crim` recibió una transformación logarítmica antes de escalarse, dado que era la variable más asimétrica del dataset.

**Escalado y Encoding:** todas las variables continuas quedaron estandarizadas (media≈0), y las categóricas (`rad`, `rad_group`) quedaron codificadas con One-Hot, generando 11 columnas adicionales a partir de solo 2 columnas originales. Esto explica el salto de 13 a 24 columnas en el resultado final.

**Conclusión general:** el pipeline de scikit-learn ejecuta correctamente los 5 pasos exigidos por la Issue 4 (limpieza, feature selection, feature engineering, escalado y encoding) de forma reproducible y sin errores. El principal punto abierto para la siguiente etapa de modelamiento es decidir el tratamiento de las observaciones censuradas (`medv_censored`) y evaluar si mantener simultáneamente `rad` y `rad_group` aporta valor o solo añade dimensionalidad innecesaria, como ya se señaló en las recomendaciones.

## 💡 Recomendaciones e ideas

**Manejo de la censura en medv:**
Recomendación: evaluar el desempeño del modelo con y sin las 11 observaciones marcadas en `medv_censored`, o entrenar un modelo separado solo para ese segmento.
Justificación: representan el 3.4 % de las 321 filas usadas (11/321), una proporción pequeña pero no despreciable; si esas observaciones no reflejan precios reales, incluirlas sin distinción puede sesgar los coeficientes de un modelo de regresión, como se discutió en el análisis multivariable.

**Multicolinealidad restante:**
Recomendación: revisar el Factor de Inflación de Varianza (VIF) sobre las columnas `numeric__*` del `x_train_transformed`, en particular para el grupo `nox`, `indus`, `age` y `dis`, que mostraron correlaciones altas entre sí en la matriz de correlación del bivariable, aunque no se descartó ninguna de estas variables en este notebook.

**Alta dimensionalidad por el One-Hot de rad:**
Recomendación: evaluar si las 9 columnas `nominal__rad_*` aportan más que las 2 columnas `nominal__rad_group_*` en el desempeño del modelo; de no ser así, considerar eliminar el `rad` original y quedarse solo con `rad_group`, ya que ambos codifican en gran medida la misma información (la distinción `rad = 24` vs el resto) y mantener las dos versiones incrementa la dimensionalidad de 24 columnas sin necesariamente aportar información nueva.

**Interacción rm–lstat:**
Recomendación: evaluar la inclusión de un término de interacción entre `rm` y `lstat` (por ejemplo, con `PolynomialFeatures(interaction_only=True)`), dado que el análisis multivariable sugirió que el efecto de `rm` sobre `medv` se atenúa cuando `lstat` es alto.

**Elección de modelo:**
Recomendación: si se prueban modelos basados en árboles (Random Forest, Gradient Boosting) además de modelos lineales, considerar que el escalado no es necesario para estos, aunque no afecta negativamente su desempeño.

## 📖 References

- Ejemplo de Feature Engineering (dataset Titanic) — Jose R. Zapata, profesor del curso.
- https://joserzapata.github.io/post/ciencia-datos-proyecto-python/4-feat_eng/
- https://scikit-learn.org/stable/modules/preprocessing.html
- https://scikit-learn.org/stable/modules/compose.html
- https://joserzapata.github.io/courses/ciencia-datos-en-produccion/control-versiones/branching-model/